# 🚀 AWS + Snowflake GenAI Workshop

This notebook demonstrates the integrated capabilities of **Snowflake Cortex** and optionally **Amazon Bedrock** for building conversational AI applications.

## What You'll Learn
- Query FOMC documents using **Cortex Search** (with Document AI parsing)
- Analyze revenue data using **Cortex Analyst** (with Semantic Views)
- Orchestrate both tools using **Cortex Agent** or **Bedrock Agent**

---

## 📦 Setup

First, let's connect to Snowflake and verify our environment.

In [ ]:
# Import required libraries
from snowflake.snowpark.context import get_active_session
import pandas as pd
import json

# Get active Snowflake session (when running in Snowflake Notebooks)
session = get_active_session()

# Set context
session.sql("USE DATABASE WORKSHOP_DB").collect()
session.sql("USE WAREHOUSE WORKSHOP_WH").collect()

print("✅ Connected to Snowflake!")
print(f"Current database: {session.get_current_database()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## 🔍 Part 1: Cortex Search (FOMC Documents)

Let's explore the FOMC documents that were parsed using **Document AI**.

In [ ]:
# Check what documents we have
docs_df = session.sql("""
    SELECT 
        FILE_NAME,
        LENGTH(EXTRACTED_TEXT) as TEXT_LENGTH,
        PARSED_AT
    FROM WORKSHOP_DB.PUBLIC.FOMC_DOCUMENTS_RAW
    ORDER BY FILE_NAME
""").to_pandas()

print("📄 FOMC Documents parsed with Document AI:")
docs_df

In [ ]:
# Check the chunks created for search
chunks_summary = session.sql("""
    SELECT 
        FILE_NAME,
        COUNT(*) as NUM_CHUNKS,
        AVG(LENGTH(CHUNK)) as AVG_CHUNK_LENGTH,
        MEETING_DATE
    FROM WORKSHOP_DB.PUBLIC.FOMC_DOCS_CHUNKS
    GROUP BY FILE_NAME, MEETING_DATE
    ORDER BY MEETING_DATE DESC
""").to_pandas()

print("📊 Document chunks for Cortex Search:")
chunks_summary

In [ ]:
# Test Cortex Search
def search_fomc(query: str, num_results: int = 3):
    results = session.sql(f"""
        SELECT * FROM TABLE(WORKSHOP_DB.PUBLIC.SEARCH_FOMC_DOCS('{query}', {num_results}))
    """).to_pandas()
    return results

print("🔎 Searching for 'inflation expectations'...")
results = search_fomc("inflation expectations")

for idx, row in results.iterrows():
    print(f"📅 Meeting: {row['MEETING_DATE']} | Score: {row['RELEVANCE_SCORE']:.3f}")
    print(f"   {row['CHUNK'][:200]}...\n")

## 📊 Part 2: Cortex Analyst (Revenue Data)

Now let's explore the revenue data using the **Semantic View**.

In [ ]:
# Preview revenue data
print("💰 Revenue Data Overview:")

daily_summary = session.sql("""
    SELECT 
        MIN(DATE) as MIN_DATE,
        MAX(DATE) as MAX_DATE,
        SUM(REVENUE) as TOTAL_REVENUE,
        SUM(COGS) as TOTAL_COGS,
        SUM(REVENUE) - SUM(COGS) as TOTAL_PROFIT
    FROM WORKSHOP_DB.REVENUE_TIMESERIES.DAILY_REVENUE
""").to_pandas()

print("Daily Revenue Summary:")
daily_summary

In [ ]:
# Revenue by region
region_summary = session.sql("""
    SELECT 
        SALES_REGION,
        SUM(REVENUE) as TOTAL_REVENUE,
        SUM(REVENUE - COGS) as PROFIT
    FROM WORKSHOP_DB.REVENUE_TIMESERIES.DAILY_REVENUE_BY_REGION
    GROUP BY SALES_REGION
    ORDER BY TOTAL_REVENUE DESC
""").to_pandas()

print("Revenue by Region:")
region_summary

In [ ]:
# Visualize revenue trends
import streamlit as st

monthly_revenue = session.sql("""
    SELECT 
        DATE_TRUNC('month', DATE) as MONTH,
        SUM(REVENUE) as REVENUE,
        SUM(REVENUE - COGS) as PROFIT
    FROM WORKSHOP_DB.REVENUE_TIMESERIES.DAILY_REVENUE
    GROUP BY 1
    ORDER BY 1
""").to_pandas()

st.subheader("📈 Monthly Revenue Trend")
st.line_chart(monthly_revenue.set_index('MONTH')[['REVENUE', 'PROFIT']])

## 🤖 Part 3: Cortex Agent

Now let's interact with the **Cortex Agent** that orchestrates both Cortex Search and Cortex Analyst.

In [ ]:
# Function to chat with the Cortex Agent
def chat_with_agent(message: str) -> str:
    result = session.sql(f"""
        CALL WORKSHOP_DB.AGENTS.CHAT_WITH_AGENT('{message.replace("'", "''")}')
    """).collect()
    
    if result:
        response_data = json.loads(result[0][0])
        return response_data.get('response', 'No response')
    return 'No response'

print("🤖 Cortex Agent is ready!")

In [ ]:
# Test Agent with FOMC question
print("📝 Question: What did the Fed say about inflation?\n")
response = chat_with_agent("What did the Fed say about inflation?")
print(f"🤖 Response: {response}")

In [ ]:
# Test Agent with Revenue question
print("📝 Question: What was total profit by region?\n")
response = chat_with_agent("What was total profit by region?")
print(f"🤖 Response: {response}")

## 💬 Part 4: Interactive Chat

Use Streamlit components for interactive exploration.

In [ ]:
import streamlit as st

st.title("�� Workshop Agent Chat")

if 'messages' not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message['role']):
        st.write(message['content'])

if prompt := st.chat_input("Ask about FOMC meetings or revenue data..."):
    st.session_state.messages.append({'role': 'user', 'content': prompt})
    with st.chat_message('user'):
        st.write(prompt)
    
    with st.chat_message('assistant'):
        with st.spinner('Thinking...'):
            response = chat_with_agent(prompt)
            st.write(response)
    
    st.session_state.messages.append({'role': 'assistant', 'content': response})

## 📚 Summary

In this notebook, you explored:

| Feature | Description |
|---------|-------------|
| **Document AI** | Native PDF parsing |
| **Cortex Search** | Vector search over documents |
| **Semantic View** | Natural language to SQL |
| **Cortex Agent** | Multi-tool orchestration |